In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
from functools import partial
from jax import flatten_util


# ==============================================================================
# 1. 全局参数 & LiH 分子定义
# ==============================================================================
# LiH 平衡键长约为 1.595 Angstrom
bond_length = 1.595
geometry = [('Li', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# 计算电子数
n_electrons = mol.nelectron
print(f"LiH 分子电子数: {n_electrons}")

# FCI 计算获取基准能量
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("LiH FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")
# 构建 Hamiltonian 和 Hilbert 空间
ha = nkx.operator.from_pyscf_molecule(mol)
# 获取轨道数并构建 Hilbert 空间
# STO-3G 基组下 LiH 有 7 个基函数 (Li: 1s,2s,2p; H: 1s) -> 14 个自旋轨道
n_orbitals = mol.nao_nr()  # 实轨道数
n_spin_orbitals = 2 * n_orbitals  # 自旋轨道数
print(f"实轨道数: {n_orbitals}, 自旋轨道数: {n_spin_orbitals}")
# 电子排布: Li(3电子) + H(1电子) = 4电子
# 分配: 2个 α 电子, 2个 β 电子
n_fermions_per_spin = (n_electrons // 2, n_electrons // 2)
print(f"每自旋电子数: {n_fermions_per_spin}")

hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=n_orbitals,
    s=1/2,
    n_fermions_per_spin=n_fermions_per_spin,
)
print(f"Hilbert 空间维度: {hi.size}")

# ==============================================================================
# 2. 神经网络 Ansatz 
# ==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=32, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear3 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        h = nnx.tanh(self.linear3(h))
        out = self.output(h)
        return jnp.squeeze(out)

# ==============================================================================
# 3. 采样器设置
# ==============================================================================
# 构建连接图 - 每个轨道作为一个节点
edges = [(0, 1),(2, 3),(4, 5),(6, 7),(8, 9),(10,11)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=100, sweep_size=32)

optimizer = nk.optimizer.Sgd(learning_rate=0.01)

# vstate = nk.vqs.MCState(sampler, model, n_samples=1008)

# gs = nk.driver.VMC(
#     ha,
#     optimizer,
#     variational_state=vstate,
#     preconditioner=nk.optimizer.SR(diag_shift=0.1,holomorphic=True),
# )

LiH 分子电子数: 4
LiH FCI 基准能量
E0 = -7.88240193 Ha  |  激发能: 0.0000 eV
E1 = -7.76641848 Ha  |  激发能: 3.1561 eV
E2 = -7.74921619 Ha  |  激发能: 3.6242 eV
E3 = -7.71645401 Ha  |  激发能: 4.5157 eV
实轨道数: 6, 自旋轨道数: 12
每自旋电子数: (2, 2)
Hilbert 空间维度: 12


In [4]:
edges

[(0, 1), (2, 3), (4, 5), (6, 7), (8, 9), (10, 11)]

In [14]:
# ===================== 4. 包装模型为 machine 函数 =====================
def create_machine(model: nnx.Module):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)
    
    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)
    
    return machine, graphdef, state

# ===================== 5. 纯 JAX 实现的 force-based 梯度计算 =====================
@partial(jax.jit, static_argnames=("machine",))
def compute_local_energies(machine, params, sigma):
    """
    计算局部能量 E_loc(σ) = Σ_η H(σ→η) ψ(η)/ψ(σ)
    """
    eta, H_eta = ha.get_conn_padded(sigma)
    logpsi_sigma = machine(params, sigma)
    logpsi_eta = machine(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_eta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)


def statistics(x):
    """计算样本统计量"""
    mean = jnp.mean(x)
    var = jnp.var(x)
    return mean, jnp.sqrt(var / x.shape[0])


@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian(machine, params, sigma):
    """
    核心：复刻 NetKet 的 forces_expect_hermitian 函数
    
    使用 force-based 梯度计算：
    ∇⟨E⟩ = ⟨(E_loc - ⟨E⟩) ∇log ψ⟩
    """
    # 1. 计算局部能量
    O_loc = compute_local_energies(machine, params, sigma)
    
    # 2. 统计能量均值
    O_mean, O_std = statistics(O_loc)
    
    # 3. 中心化局部能量
    O_centered = O_loc - O_mean
    
    # 4. 计算 ∇log ψ 对每个样本
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # 5. 计算 force-based 梯度
    def weight_and_mean(grad_component):
        weights = O_centered.reshape((O_centered.shape[0],) + (1,) * (grad_component.ndim - 1))
        return jnp.mean(weights * jnp.conj(grad_component), axis=0)
    
    grad = jax.tree_util.tree_map(weight_and_mean, grad_matrix)
    
    return O_mean, O_std, grad


def compute_qgt(machine, params, sigma, diag_shift=0.1):
    """
    计算量子几何张量（QGT）/ F 矩阵
    
    QGT 定义：
    S_ij = ⟨∂_i log ψ* ∂_j log ψ⟩ - ⟨∂_i log ψ*⟩⟨∂_j log ψ⟩
    """
    n_samples = sigma.shape[0]
    
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    grad_flat, unravel_fn = flatten_util.ravel_pytree(grad_matrix)
    grad_flat = grad_flat.reshape(n_samples, -1)
    
    grad_mean = jnp.mean(grad_flat, axis=0, keepdims=True)
    grad_centered = grad_flat - grad_mean
    
    qgt = (1.0 / n_samples) * jnp.conj(grad_centered).T @ grad_centered
    qgt_reg = qgt + diag_shift * jnp.eye(qgt.shape[0])
    
    return qgt_reg, unravel_fn

In [15]:
hi.all_states()[0]

Array([0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1], dtype=int8)

In [16]:
# ===================== 6. 初始化 =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(n_spin_orbitals, hidden_dim=32, rngs=rngs)
machine, graphdef, params = create_machine(model)
sampler_state = sampler.init_state(machine, params, seed=1)

In [18]:
energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)

In [19]:
energy

Array(nan+nanj, dtype=complex128)

In [20]:
# ===================== 6. 初始化 =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(n_spin_orbitals, hidden_dim=32, rngs=rngs)
machine, graphdef, params = create_machine(model)
sampler_state = sampler.init_state(machine, params, seed=1)

optimizer = optax.sgd(learning_rate=0.005)  # 学习率 0.005
opt_state = optimizer.init(params)

# 训练参数
N_ITER = 500  # 迭代次数
N_SAMPLES = 1008  # 样本数

# ===================== 7. 训练循环 =====================
print("\n" + "="*60)
print("开始纯 JAX VMC 训练 (自然梯度下降法) - LiH 分子")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}

for step in range(N_ITER):
    # 1. 采样
    sampler_state = sampler.reset(machine, params, sampler_state)
    
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, 
        chain_length=20
    )
    samples = samples.reshape(-1, hi.size)
    
    # 2. 计算 force-based 能量和梯度
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    grad = jax.tree_util.tree_map(lambda x: x*2, grad)
    
    # # 3. 计算 QGT 并求自然梯度
    # qgt_reg, qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.01)
    # grad_flat, grad_unravel_fn = flatten_util.ravel_pytree(grad)
  
    # # 自然梯度 natural-gradient = S^{-1} * grad
    # natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    # natural_grad = grad_unravel_fn(natural_grad)
    # grad = natural_grad
        
    # 4. 更新参数（自然梯度下降）
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 记录历史
    if step % 1 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

# 最终结果
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)


开始纯 JAX VMC 训练 (自然梯度下降法) - LiH 分子
Step   0 | E: -3.74631026 ± 0.045590 | FCI: -7.88240193 | Error: 4.136092
Step   1 | E: -4.09188756 ± 0.056446 | FCI: -7.88240193 | Error: 3.790514
Step   2 | E: -3.71914736 ± 0.047916 | FCI: -7.88240193 | Error: 4.163255
Step   3 | E: -2.04978450 ± 0.250457 | FCI: -7.88240193 | Error: 5.832617
Step   4 | E: -5.13975949 ± 0.253023 | FCI: -7.88240193 | Error: 2.742642
Step   5 | E: -463156073475606.93750000 ± 360923493217183.125000 | FCI: -7.88240193 | Error: 463156073475599.062500
Step   6 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step   7 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step   8 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step   9 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step  10 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step  11 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step  12 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step  13 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step  14 | E: nan ± nan | FCI: 

KeyboardInterrupt: 